In [1]:
# Standard library
import datetime
import gc
import logging
import os
import sys
from collections import defaultdict
from glob import glob
from typing import Dict, TypedDict, Union

# Third-party
import numpy as np
import optuna
import pandas as pd
import torch
import torch.distributed as dist
import torch.utils.data
from torch.cuda.amp import autocast
from torch.utils.data import DistributedSampler, IterableDataset, get_worker_info
from torchvision import transforms as tforms
import tqdm
import xarray as xr
import yaml

# Local application
from credit.data import WRF_Dataset, concat_and_reshape, reshape_only
from credit.models.checkpoint import TorchFSDPCheckpointIO
from credit.parser import credit_main_parser, training_data_check
#from credit.postblock import GlobalEnergyFixer, GlobalMassFixer, GlobalWaterFixer
from credit.scheduler import update_on_batch, update_on_epoch
from credit.trainers.base_trainer import BaseTrainer
from credit.trainers.utils import accum_log, cleanup, cycle
from credit.transforms import Normalize_WRF, ToTensor_WRF

logger = logging.getLogger(__name__)

In [2]:
# from credit.data import WRF_Dataset
# from credit.transforms import Normalize_WRF, ToTensor_WRF

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
# from credit.data import (
#     generate_datetime,
#     hour_to_nanoseconds,
#     nanoseconds_to_year,
#     extract_month_day_hour,
#     find_common_indices,
#     concat_and_reshape,
#     reshape_only,
#     get_forward_data,
#     drop_var_from_dataset,
#     find_key_for_number
# )

In [5]:
# Logging setup
logger = logging.getLogger(__name__)

# single node steup
rank = 0
world_size = 1

In [6]:
config_name = '/glade/work/ksha/DWC_runs/CONUS_GP_base/model_single.yml'
# Read YAML file
with open(config_name, 'r') as stream:
    conf = yaml.safe_load(stream)

In [7]:
conf = credit_main_parser(conf, parse_training=True, parse_predict=False, print_summary=True)

Upper-air variables: ['WRF_P', 'WRF_U', 'WRF_V', 'WRF_T', 'WRF_Q_tot_05']
Surface variables: ['WRF_SP', 'WRF_MSLP', 'WRF_T2', 'WRF_TD2', 'WRF_U10', 'WRF_V10', 'WRF_PWAT_05', 'WRF_SMOIS', 'WRF_TSLB']
Dynamic forcing variables: []
Diagnostic variables: []
Forcing variables: []
Static variables: ['z_norm', 'var_norm', 'lu_norm', 'LANDMASK']


## dataset dev

In [8]:
Array = Union[np.ndarray, xr.DataArray]

class Sample(TypedDict):
    # Shape: batch_size, seq_length, lat, lon, lev
    WRF_input: Array
    WRF_target: Array
    boundary_input: Array
    time_encode: Array
    datetime_index: Array

In [9]:
# pick a year
train_years_range = [2018, 2020]
valid_years_range = [2018, 2020]

# ============================================================================== #
# WRF domain interior
# ---------------------------
# param_interior
# param_outside
# ----------------------------
#     varname_upper_air
#     varname_surface
#     varname_dyn_forcing
#     varname_forcing
#     varname_static
#     varname_diagnostic
#     filenames
#     filename_surface
#     filename_dyn_forcing
#     filename_forcing
#     filename_static
#     filename_diagnostic
#     history_len
#     forecast_len
# ============================================================================== #

param_interior = {}
param_outside = {}
# --------------- #
# upper air files
upper_files = sorted(glob(conf["data"]["save_loc"]))
upper_files_outside = sorted(glob(conf["data"]['boundary']["save_loc"]))

# --------------- #
# surface files
if ('surface_variables' in conf['data']) and (len(conf['data']['surface_variables']) > 0):
    list_surf_ds = sorted(glob(conf["data"]["save_loc_surface"]))
else:
    list_surf_ds = None

list_surf_ds_outside = sorted(glob(conf["data"]['boundary']["save_loc_surface"]))

# --------------- #
# dyn forcing files
if ('dynamic_forcing_variables' in conf['data']) and (len(conf['data']['dynamic_forcing_variables']) > 0):
    list_dyn_forcing_ds = sorted(glob(conf["data"]["save_loc_dynamic_forcing"]))
else:
    list_dyn_forcing_ds = None

# --------------- #
# diagnostic files
if ('diagnostic_variables' in conf['data']) and (len(conf['data']['diagnostic_variables']) > 0):
    list_diag_ds = sorted(glob(conf["data"]["save_loc_diagnostic"]))
else:
    list_diag_ds = None

# convert year info to str for file name search
train_years = [str(year) for year in range(train_years_range[0], train_years_range[1])]
valid_years = [str(year) for year in range(valid_years_range[0], valid_years_range[1])]

# Filter the files for training / validation
train_files = [file for file in upper_files if any(year in file for year in train_years)]
valid_files = [file for file in upper_files if any(year in file for year in valid_years)]

train_files_outside = [file for file in upper_files_outside if any(year in file for year in train_years)]
valid_files_outside = [file for file in upper_files_outside if any(year in file for year in valid_years)]

if list_surf_ds is not None:
    train_list_surf_ds = [file for file in list_surf_ds if any(year in file for year in train_years)]
    valid_list_surf_ds = [file for file in list_surf_ds if any(year in file for year in valid_years)]
else:
    train_list_surf_ds = None
    valid_list_surf_ds = None

train_list_surf_ds_outside = [file for file in list_surf_ds_outside if any(year in file for year in train_years)]
valid_list_surf_ds_outside = [file for file in list_surf_ds_outside if any(year in file for year in valid_years)]

if list_dyn_forcing_ds is not None:
    train_list_dyn_forcing_ds = [file for file in list_dyn_forcing_ds if any(year in file for year in train_years)]
    valid_list_dyn_forcing_ds = [file for file in list_dyn_forcing_ds if any(year in file for year in valid_years)]

else:
    train_list_dyn_forcing_ds = None
    valid_list_dyn_forcing_ds = None

if list_diag_ds is not None:
    train_list_diag_ds = [file for file in list_diag_ds if any(year in file for year in train_years)]
    valid_list_diag_ds = [file for file in list_diag_ds if any(year in file for year in valid_years)]
else:
    train_list_diag_ds = None
    valid_list_diag_ds = None
    
param_interior['varname_upper_air'] = conf['data']['variables']
param_interior['varname_surface'] = conf['data']['surface_variables']
param_interior['varname_dyn_forcing'] = conf['data']['dynamic_forcing_variables']
param_interior['varname_forcing'] = conf['data']['forcing_variables']
param_interior['varname_static'] = conf['data']['static_variables']
param_interior['varname_diagnostic'] = conf['data']['diagnostic_variables']
param_interior['filename_forcing'] = conf['data']['save_loc_forcing']
param_interior['filename_static'] = conf['data']['save_loc_static']

param_outside['varname_upper_air'] = conf["data"]['boundary']['variables']
param_outside['varname_surface'] = conf["data"]['boundary']['surface_variables']
# --------------------------------------------------- #
is_train = False

# separate training set and validation set cases
if is_train:
    param_interior['filenames'] = train_files
    param_interior['filename_surface'] = train_list_surf_ds
    param_interior['filename_dyn_forcing'] = train_list_dyn_forcing_ds
    param_interior['filename_diagnostic'] = train_list_diag_ds
    param_interior['history_len'] = conf["data"]["history_len"]
    param_interior['forecast_len'] = conf["data"]["forecast_len"]

    param_outside['filenames'] = train_files_outside
    param_outside['filename_surface'] = train_list_surf_ds_outside
    param_outside['history_len'] = conf["data"]['boundary']["history_len"]
    param_outside['forecast_len'] = conf["data"]['boundary']["forecast_len"]
    
    name = "training"
    
else:
    param_interior['filenames'] = valid_files
    param_interior['filename_surface'] = valid_list_surf_ds
    param_interior['filename_dyn_forcing'] = valid_list_dyn_forcing_ds
    param_interior['filename_diagnostic'] = valid_list_diag_ds
    param_interior['history_len'] = conf["data"]["valid_history_len"]
    param_interior['forecast_len'] = conf["data"]["valid_forecast_len"]
    
    param_outside['filenames'] = valid_files_outside
    param_outside['filename_surface'] = valid_list_surf_ds_outside
    param_outside['history_len'] = conf["data"]['boundary']["history_len"]
    param_outside['forecast_len'] = conf["data"]['boundary']["forecast_len"]
    
    name = 'validation'
    

In [10]:
to_tensor_scaler = ToTensor_WRF(conf)
normalizer = Normalize_WRF(conf)
transforms = tforms.Compose([normalizer, to_tensor_scaler])

In [11]:
dataset = WRF_Dataset(
    param_interior,
    param_outside,
    transform=None,
)

In [12]:
batch_single = dataset.__getitem__(333)

In [13]:
batch_single.keys()

dict_keys(['WRF_input', 'WRF_target', 'boundary_input', 'time_encode', 'datetime_index', 'index'])

In [14]:
batch_single['boundary_input']

<xarray.Dataset>
Dimensions:      (time: 2, level: 11, south_north: 336, west_east: 336)
Coordinates:
  * level        (level) float64 1e+03 950.0 850.0 700.0 ... 200.0 100.0 50.0
  * south_north  (south_north) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
  * time         (time) datetime64[ns] 2018-01-14T21:00:00 2018-01-15
  * west_east    (west_east) float32 0.0 1.0 2.0 3.0 ... 332.0 333.0 334.0 335.0
Data variables:
    Q            (time, level, south_north, west_east) float32 dask.array<chunksize=(1, 11, 336, 336), meta=np.ndarray>
    T            (time, level, south_north, west_east) float32 dask.array<chunksize=(1, 11, 336, 336), meta=np.ndarray>
    U            (time, level, south_north, west_east) float32 dask.array<chunksize=(1, 11, 336, 336), meta=np.ndarray>
    V            (time, level, south_north, west_east) float32 dask.array<chunksize=(1, 11, 336, 336), meta=np.ndarray>
    MSL          (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    VAR_10U      (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    VAR_10V      (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    VAR_2T       (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
Attributes:
    regrid_method:  bilinear

In [15]:
batch_single['WRF_input']

<xarray.Dataset>
Dimensions:       (time: 1, bottom_top: 12, south_north: 336, west_east: 336)
Coordinates:
  * bottom_top    (bottom_top) float32 0.0 1.0 2.0 3.0 4.0 ... 8.0 9.0 10.0 11.0
  * south_north   (south_north) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
  * time          (time) datetime64[ns] 2018-01-14T21:00:00
  * west_east     (west_east) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
Data variables: (12/18)
    WRF_P         (time, bottom_top, south_north, west_east) float32 8.845e+0...
    WRF_U         (time, bottom_top, south_north, west_east) float32 0.4499 ....
    WRF_V         (time, bottom_top, south_north, west_east) float32 -0.2784 ...
    WRF_T         (time, bottom_top, south_north, west_east) float32 288.1 .....
    WRF_Q_tot_05  (time, bottom_top, south_north, west_east) float32 0.05817 ...
    WRF_SP        (time, south_north, west_east) float32 8.872e+04 ... 1.012e+05
    ...            ...
    WRF_SMOIS     (time, south_north, west_east) float32 0.2281 ... 0.2988
    WRF_TSLB      (time, south_north, west_east) float32 293.4 294.4 ... 272.8
    LANDMASK      (time, south_north, west_east) float32 1.0 1.0 1.0 ... 1.0 1.0
    lu_norm       (time, south_north, west_east) float32 0.4118 ... 0.7059
    var_norm      (time, south_north, west_east) float32 0.7052 ... -0.3643
    z_norm        (time, south_north, west_east) float32 1.239 1.313 ... -0.407

In [16]:
batch_single['WRF_target']

<xarray.Dataset>
Dimensions:       (time: 1, bottom_top: 12, south_north: 336, west_east: 336)
Coordinates:
  * bottom_top    (bottom_top) float32 0.0 1.0 2.0 3.0 4.0 ... 8.0 9.0 10.0 11.0
  * south_north   (south_north) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
  * time          (time) datetime64[ns] 2018-01-14T22:00:00
  * west_east     (west_east) float32 0.0 1.0 2.0 3.0 ... 333.0 334.0 335.0
Data variables: (12/14)
    WRF_P         (time, bottom_top, south_north, west_east) float32 8.842e+0...
    WRF_U         (time, bottom_top, south_north, west_east) float32 0.1297 ....
    WRF_V         (time, bottom_top, south_north, west_east) float32 -0.2716 ...
    WRF_T         (time, bottom_top, south_north, west_east) float32 289.1 .....
    WRF_Q_tot_05  (time, bottom_top, south_north, west_east) float32 0.05792 ...
    WRF_SP        (time, south_north, west_east) float32 8.869e+04 ... 1.011e+05
    ...            ...
    WRF_TD2       (time, south_north, west_east) float32 271.3 269.8 ... 253.2
    WRF_U10       (time, south_north, west_east) float32 0.1222 ... -1.324
    WRF_V10       (time, south_north, west_east) float32 -0.255 ... 3.435
    WRF_PWAT_05   (time, south_north, west_east) float32 0.07247 ... 0.0803
    WRF_SMOIS     (time, south_north, west_east) float32 0.2274 ... 0.2986
    WRF_TSLB      (time, south_north, west_east) float32 294.1 295.2 ... 272.8

In [14]:
print(batch_single['x'].shape)
print(batch_single['x_boundary'].shape)
print(batch_single['x_surf_boundary'].shape)
print(batch_single['x_time_encode'].shape)

torch.Size([1, 5, 12, 336, 336])
torch.Size([2, 4, 11, 336, 336])
torch.Size([2, 4, 336, 336])
torch.Size([16])


In [15]:
batch = {}
keys = list(batch_single.keys())
keys = keys[:-1]

for var in keys:
    batch[var] = batch_single[var].unsqueeze(0) # give a single sample batch dimension

# ------------------------- #
# base trainer workflow

if "x_surf" in batch:
    # combine x and x_surf
    # input: (batch_num, time, var, level, lat, lon), (batch_num, time, var, lat, lon)
    # output: (batch_num, var, time, lat, lon), 'x' first and then 'x_surf'
    x = concat_and_reshape(batch["x"], batch["x_surf"])
else:
    # no x_surf
    x = reshape_only(batch["x"]).to(self.device)

# --------------------------------------------------------------------------------- #
# add forcing and static variables
if 'x_forcing_static' in batch:

    # (batch_num, time, var, lat, lon) --> (batch_num, var, time, lat, lon)
    x_forcing_batch = batch['x_forcing_static'].permute(0, 2, 1, 3, 4)

    # concat on var dimension
    x = torch.cat((x, x_forcing_batch), dim=1)

# --------------------------------------------------------------------------------- #
# combine y and y_surf
if "y_surf" in batch:
    y = concat_and_reshape(batch["y"], batch["y_surf"])
else:
    y = reshape_only(batch["y"])

if 'y_diag' in batch:

    # (batch_num, time, var, lat, lon) --> (batch_num, var, time, lat, lon)
    y_diag_batch = batch['y_diag'].permute(0, 2, 1, 3, 4).float()

    # concat on var dimension
    y = torch.cat((y, y_diag_batch), dim=1)

In [16]:
x_time_encode = batch['x_time_encode']

In [17]:
x_time_encode.shape

torch.Size([1, 16])

In [18]:
x.shape

torch.Size([1, 73, 1, 336, 336])

In [19]:
y.shape

torch.Size([1, 69, 1, 336, 336])

**Multistep training fix**

credit.dataset era5_multistep  ERA5_and_Forcing_MultiStep

## boundary and interior data

In [5]:
ds_ERA5 = xr.open_zarr('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/all_in_one/ERA5_GP_1980.zarr')

In [7]:
ds_ERA5

<xarray.Dataset>
Dimensions:    (time: 1464, latitude: 88, longitude: 112, level: 11)
Coordinates:
  * latitude   (latitude) float64 45.0 44.75 44.5 44.25 ... 23.75 23.5 23.25
  * level      (level) float64 50.0 100.0 200.0 300.0 ... 850.0 950.0 1e+03
  * longitude  (longitude) float64 250.2 250.5 250.8 251.0 ... 277.5 277.8 278.0
  * time       (time) datetime64[ns] 1980-01-01 ... 1980-12-31T18:00:00
Data variables:
    MSL        (time, latitude, longitude) float32 dask.array<chunksize=(4, 88, 112), meta=np.ndarray>
    Q          (time, level, latitude, longitude) float32 dask.array<chunksize=(4, 2, 88, 112), meta=np.ndarray>
    SP         (time, latitude, longitude) float32 dask.array<chunksize=(4, 88, 112), meta=np.ndarray>
    T          (time, level, latitude, longitude) float32 dask.array<chunksize=(4, 2, 88, 112), meta=np.ndarray>
    U          (time, level, latitude, longitude) float32 dask.array<chunksize=(4, 2, 88, 112), meta=np.ndarray>
    V          (time, level, latitude, longitude) float32 dask.array<chunksize=(4, 2, 88, 112), meta=np.ndarray>
    VAR_10U    (time, latitude, longitude) float32 dask.array<chunksize=(4, 88, 112), meta=np.ndarray>
    VAR_10V    (time, latitude, longitude) float32 dask.array<chunksize=(4, 88, 112), meta=np.ndarray>
    VAR_2T     (time, latitude, longitude) float32 dask.array<chunksize=(4, 88, 112), meta=np.ndarray>
    Z          (time, level, latitude, longitude) float32 dask.array<chunksize=(4, 2, 88, 112), meta=np.ndarray>
Attributes:
    CONVERSION_DATE:      Sun May 19 19:53:54 MDT 2019
    CONVERSION_PLATFORM:  Linux r1i0n9 3.12.62-60.64.8-default #1 SMP Tue Oct...
    Conventions:          CF-1.6
    DATA_SOURCE:          ECMWF: https://cds.climate.copernicus.eu, Copernicu...
    NCO:                  netCDF Operators version 4.7.4 (http://nco.sf.net)
    NETCDF_COMPRESSION:   NCO: Precision-preserving compression to netCDF4/HD...
    NETCDF_CONVERSION:    CISL RDA: Conversion from ECMWF GRIB 1 data to netC...
    NETCDF_VERSION:       4.6.1
    history:              Sun May 19 19:54:09 2019: ncks -4 --ppc default=7 e...

In [6]:
ds_C404 = xr.open_zarr('/glade/campaign/ral/hap/ksha/DWC_data/CONUS_domain_GP/all_in_one/C404_GP_1980.zarr')

In [8]:
ds_C404

<xarray.Dataset>
Dimensions:                    (time: 8784, south_north: 336, west_east: 336,
                                bottom_top: 12)
Coordinates:
  * time                       (time) datetime64[ns] 1980-01-01 ... 1980-12-3...
Dimensions without coordinates: south_north, west_east, bottom_top
Data variables: (12/14)
    WRF_MSLP                   (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_Q                      (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(1, 12, 336, 336), meta=np.ndarray>
    WRF_SP                     (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_T                      (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(1, 12, 336, 336), meta=np.ndarray>
    WRF_T2                     (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_TD2                    (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    ...                         ...
    WRF_V                      (time, bottom_top, south_north, west_east) float32 dask.array<chunksize=(1, 12, 336, 336), meta=np.ndarray>
    WRF_V10                    (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_column_total_moisture  (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_evapor                 (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_precip                 (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>
    WRF_radar_composite        (time, south_north, west_east) float32 dask.array<chunksize=(1, 336, 336), meta=np.ndarray>

In [7]:
ds_static = xr.open_zarr('/glade/derecho/scratch/ksha/DWC_data/CONUS_domain_GP/static/C404_GP_static.zarr')

In [8]:
ds_static

<xarray.Dataset>
Dimensions:   (south_north: 336, west_east: 336)
Dimensions without coordinates: south_north, west_east
Data variables:
    HGT_M     (south_north, west_east) float32 dask.array<chunksize=(336, 336), meta=np.ndarray>
    LANDMASK  (south_north, west_east) float32 dask.array<chunksize=(336, 336), meta=np.ndarray>
    XLAT      (south_north, west_east) float32 dask.array<chunksize=(336, 336), meta=np.ndarray>
    XLONG     (south_north, west_east) float32 dask.array<chunksize=(336, 336), meta=np.ndarray>
    z_norm    (south_north, west_east) float32 dask.array<chunksize=(336, 336), meta=np.ndarray>
Attributes: (12/47)
    BOTTOM-TOP_GRID_DIMENSION:       0
    CEN_LAT:                         39.100006103515625
    CEN_LON:                         -97.89999389648438
    DX:                              4000.0
    DY:                              4000.0
    DYN_OPT:                         2
    ...                              ...
    j_parent_end:                    1016
    j_parent_start:                  1
    parent_grid_ratio:               1
    parent_id:                       1
    sr_x:                            1
    sr_y:                            1